# ARC-AGI-3 SG (CNN bandit) Submission

Same gateway architecture as `submit_bfs.ipynb` — see `docs/kaggle-submission.md` for details.

Strategy here: baseline-1 SG-port (CNN bandit predicting which actions cause frame changes, click prior with static-mask + 4-conn segment equalization). Uses `scripts/benchmark.py`. Per-game time budget is the natural knob.

Reference point: the official `inversion/arc3-sample-submission-stochastic-goose` (CNN bandit, no priors) scores **0.25** on LB. Our port adds the static-mask + segment priors — expect a similar or slightly higher number.

## 1. Install arc-agi SDK + torch (already preinstalled; we just need cloudpickle for tooling parity)

In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

## 2. Bring in our repo from a Kaggle dataset

In [ ]:
import os, sys, shutil, glob

REPO_DATASET_CANDIDATES = [
    "/kaggle/input/arc-agi-3-pre",
    "/kaggle/input/arc-agi-3-pre-repo",
    "/kaggle/input/arc-agi-3-pre-bfs",
    "/kaggle/input/arc-agi-3-pre-v1",
    "/kaggle/input/arc-agi-3-pre-code",
]
if not any(os.path.isdir(p) for p in REPO_DATASET_CANDIDATES):
    for cand in glob.glob("/kaggle/input/*"):
        if os.path.isfile(os.path.join(cand, "scripts/benchmark.py")):
            REPO_DATASET_CANDIDATES.insert(0, cand)
            break

REPO_SRC = next((p for p in REPO_DATASET_CANDIDATES if os.path.isdir(p)), None)
REPO_DIR = "/kaggle/working/arc-agi-3-pre"

print("-- /kaggle/input contents --")
!ls /kaggle/input/ 2>/dev/null
print()

if REPO_SRC is None:
    raise SystemExit(
        f"Could not find our repo at any of {REPO_DATASET_CANDIDATES}. "
        f"Attach the dataset to this notebook and add its path above."
    )
print(f"using repo source: {REPO_SRC}")
if not os.path.isdir(REPO_DIR):
    shutil.copytree(REPO_SRC, REPO_DIR)
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f"repo at {REPO_DIR}")

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"torch device = {DEVICE} ({torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'})")

## 3. Run SG bandit against the gateway (rerun-mode only)

Per-game `--budget-min`: leave ~1h slack vs the 12h cap. For ~100 games / 11h budget → 6-7 min per game.

In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    os.environ['ARC_BASE_URL']   = 'http://gateway:8001/'
    os.environ['ARC_API_KEY']    = 'test-key-123'
    os.environ['OPERATION_MODE'] = 'online'

    import torch
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Budget reasoning (post-mortem of v1 timeout):
    #   - v1 used --budget-min 6, never closed scorecard → 0 LB
    #   - --total-budget-min 360 hard caps the run, --budget-min 3 per game
    #   - benchmark.py will further cap per-game by remaining-budget so the last
    #     game doesn't blow past total_deadline
    !cd /kaggle/working/arc-agi-3-pre && python scripts/benchmark.py \
        --device {DEVICE} \
        --budget-min 3 \
        --total-budget-min 360 \
        --tag kaggle-sg-v2 \
        --segment-prior \
        --log-every 200
else:
    print("Not in competition rerun — skipping SG run. (Will write dummy submission in next cell.)")

## 4. Edit-mode dummy submission

In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    print(submission)